In [ ]:
import rarfile

# Если unrar.exe не в PATH, укажите путь явно (для Windows)
rarfile.UNRAR_TOOL = r'C:\Program Files\WinRAR\UnRAR.exe'  # укажите ваш путь

# Распаковка архива
with rarfile.RarFile('1.rar') as rf:
    rf.extractall('./')
    print("Архив успешно распакован!")

Архив успешно распакован!


In [ ]:
def process_file(input_file, output_file, ecg_fs=1000, cam_fps=100):
    """
    Присваивает номера кадров каждому отсчету ЭКГ.
    - 1 кадр камеры = 10 отсчетов ЭКГ
    - При нарушении синхронизации кадр пустой
    - После восстановления синхронизации начинается новый кадр
    """
    try:
        samples_per_frame = ecg_fs // cam_fps
        frame_number = 1
        counter_in_frame = 0
        sync_active = True

        with open(input_file, 'r', encoding='utf-8') as infile, \
             open(output_file, 'w', encoding='utf-8') as outfile:

            for line in infile:
                columns = line.strip().split()
                if len(columns) < 10:
                    continue

                second_column = columns[1]
                tenth_column = columns[9]

                if tenth_column == "0x00000000":
                    # Потеря синхронизации
                    sync_active = False
                    current_frame = ""
                    counter_in_frame = 0  # обнуляем счетчик
                else:
                    if not sync_active:
                        # Восстановление синхронизации — новый кадр
                        sync_active = True
                        frame_number += 1
                        counter_in_frame = 1
                        current_frame = frame_number
                    else:
                        # Продолжаем текущий кадр
                        counter_in_frame += 1
                        current_frame = frame_number

                    # Каждые 10 отсчетов — новый кадр
                    if counter_in_frame >= samples_per_frame:
                        frame_number += 1
                        counter_in_frame = 0

                outfile.write(f"{second_column} {tenth_column} {current_frame}\n")

        print(f"Файл успешно обработан. Результат сохранен в: {output_file}")
        print(f"Всего пронумеровано кадров: {frame_number - 1}")

    except FileNotFoundError:
        print(f"Ошибка: Файл '{input_file}' не найден.")
    except Exception as e:
        print(f"Произошла ошибка: {e}")

import numpy as np
from scipy.signal import butter, filtfilt
import matplotlib.pyplot as plt
from matplotlib.widgets import Cursor, Button
import matplotlib as mpl

def read_ecg_data(filename):
    # Чтение данных
    ecg_data = []
    frame_numbers = []

    with open(filename, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) >= 3:
                ecg_value = int(parts[0], 16)  # ЭКГ
                frame_num = int(parts[2])  # номер кадра

                ecg_data.append(ecg_value)
                frame_numbers.append(frame_num)

    return np.array(ecg_data), np.array(frame_numbers)


def preprocess_ecg_signal(ecg_signal):
    # Нормализация ЭКГ
    # Удаление постоянной составляющей
    ecg_centered = ecg_signal - np.mean(ecg_signal)

    # Нормализация
    max_abs = np.max(np.abs(ecg_centered))

    if max_abs > 0:
        ecg_normalized = ecg_centered / max_abs
    else:
        ecg_normalized = ecg_centered

    return ecg_normalized


def pan_tompkins(ecg_signal, fs=1000):
    # Фильтрация
    nyquist = 0.5 * fs
    low = 10.0 / nyquist
    high = 40.0 / nyquist
    b, a = butter(1, [low, high], btype='band')
    filtered_ecg = filtfilt(b, a, ecg_signal)

    # Дифференцирование
    diff_ecg = np.diff(filtered_ecg)
    diff_ecg = np.append(diff_ecg, diff_ecg[-1])

    # Возведение в квадрат
    squared_ecg = diff_ecg ** 2

    # Интегрирование
    window_size = int(0.08 * fs)
    integrated_ecg = np.convolve(squared_ecg, np.ones(window_size) / window_size, mode='same')

    # Поиск пиков
    r_peaks_indices = []
    refractory_period = int(0.1 * fs)  # рефрактерный период

    # Порог на основе процентиля
    threshold = np.percentile(integrated_ecg, 0)

    for i in range(1, len(integrated_ecg) - 1):
        # Проверка рефрактерного периода
        if len(r_peaks_indices) > 0 and (i - r_peaks_indices[-1]) < refractory_period:
            continue

        if (integrated_ecg[i] > threshold and
                integrated_ecg[i] > integrated_ecg[i - 1] and
                integrated_ecg[i] > integrated_ecg[i + 1]):

            search_window = int(0.04 * fs)
            search_start = max(0, i - search_window)
            search_end = min(len(ecg_signal) - 1, i + search_window)
            exact_peak = np.argmax(ecg_signal[search_start:search_end]) + search_start

            r_peaks_indices.append(exact_peak)

    return r_peaks_indices


def validate_r_peaks(r_peaks_indices, fs=1000):

    if len(r_peaks_indices) < 2:
        return r_peaks_indices

    valid_peaks = [r_peaks_indices[0]]

    min_rr_interval = int(0.06 * fs)  # max ЧСС
    max_rr_interval = int(0.15 * fs)  # min ЧСС

    for i in range(1, len(r_peaks_indices)):
        rr_interval = r_peaks_indices[i] - r_peaks_indices[i - 1]

        if min_rr_interval <= rr_interval <= max_rr_interval:
            valid_peaks.append(r_peaks_indices[i])

    return valid_peaks


def detect_r_peaks_from_file(filename, fs=1000):
    # Чтение данных
    ecg_signal, frame_numbers = read_ecg_data(filename)

    print(f"Прочитано {len(ecg_signal)} отсчетов ECG")
    print(f"Частота дискретизации: {fs} Гц")

    # Предобработка сигнала
    ecg_signal = preprocess_ecg_signal(ecg_signal)

    # Детекция R-пиков
    r_peaks_indices = pan_tompkins(ecg_signal, fs)

    # Валидация результатов
    r_peaks_indices = validate_r_peaks(r_peaks_indices, fs)

    # Получение номеров кадров
    r_peak_frames = frame_numbers[r_peaks_indices]

    # Вывод результатов
    print(f"\n Найдено {len(r_peaks_indices)} R-пиков:")
    print("Номера кадров R-пиков:")
    for i, (frame, index) in enumerate(zip(r_peak_frames, r_peaks_indices)):
        time_sec = index / fs
        rr_interval = "N/A" if i == 0 else f"{(r_peaks_indices[i] - r_peaks_indices[i - 1]) / fs * 1000:.1f} мс"
        print(f"R-пик {i + 1:2d}: кадр {frame}, время {time_sec:.3f} сек, RR: {rr_interval}")

    return r_peak_frames, r_peaks_indices, ecg_signal, frame_numbers


def save_r_peaks_to_file(r_peaks_indices, frame_numbers, fs=1000, filename="r_peaks_results_mouse.txt"):
    """Сохраняет R-пики в файл"""
    r_peak_frames = frame_numbers[r_peaks_indices]
    
    with open(filename, "w") as f:
        for i, frame in enumerate(r_peak_frames):
            time_sec = r_peaks_indices[i] / fs
            f.write(f"{frame}\t{time_sec:.3f}\n")
    
    print(f"Результаты сохранены в файл '{filename}'")
    return r_peak_frames


class InteractiveRPeakEditor:
    def __init__(self, ecg_signal, r_peaks_indices, frame_numbers, fs=1000, start_second=5, end_second=10):
        self.ecg_signal = ecg_signal
        self.r_peaks_indices = r_peaks_indices.copy()
        self.frame_numbers = frame_numbers
        self.fs = fs
        self.start_second = start_second
        self.end_second = end_second
        self.edit_mode = True  # Режим редактирования включен по умолчанию
        self.dragging = False  # Флаг перетаскивания
        self.last_click_time = 0  # Для определения двойного клика
        
        self.fig, self.ax = plt.subplots(figsize=(15, 10))
        self.time = np.arange(len(ecg_signal)) / fs
        
        # Создаем область для кнопок
        plt.subplots_adjust(bottom=0.15)
        
        # Вычисляем диапазон отображения
        self.start_index = max(0, int(start_second * fs))
        self.end_index = min(len(ecg_signal), int(end_second * fs))
        
        self.display_time = self.time[self.start_index:self.end_index]
        self.display_signal = ecg_signal[self.start_index:self.end_index]
        self.display_frames = frame_numbers[self.start_index:self.end_index]
        
        # Отображаем сигнал
        self.line, = self.ax.plot(self.display_time, self.display_signal,
                                 label='ECG сигнал', linewidth=1.5, color='blue')
        
        # Отображаем R-пики
        self.update_r_peaks_display()
        
        # Настройка курсора
        self.cursor = Cursor(self.ax, useblit=True, color='red', linewidth=1)
        
        # Аннотация для отображения координат
        self.annot = self.ax.annotate("", xy=(0,0), xytext=(20,20), textcoords="offset points",
                            bbox=dict(boxstyle="round", fc="w", alpha=0.8),
                            arrowprops=dict(arrowstyle="->"))
        self.annot.set_visible(False)
        
        # Текст с координатами
        self.coord_text = self.ax.text(0.02, 0.98, "", transform=self.ax.transAxes, 
                                      verticalalignment='top',
                                      bbox=dict(boxstyle="round", facecolor='wheat', alpha=0.8))
        self.coord_text.set_visible(False)
        
        # Информация о R-пиках и режиме
        self.info_text = self.ax.text(0.98, 0.98, "", transform=self.ax.transAxes,
                                     verticalalignment='top', horizontalalignment='right',
                                     bbox=dict(boxstyle="round", facecolor='lightblue', alpha=0.8))
        self.update_info_text()
        
        # Создаем кнопки
        self.create_buttons()
        
        # Подключаем обработчики событий
        self.fig.canvas.mpl_connect("motion_notify_event", self.on_motion)
        self.fig.canvas.mpl_connect("axes_leave_event", self.on_leave)
        self.fig.canvas.mpl_connect('button_press_event', self.on_click)
        self.fig.canvas.mpl_connect('button_release_event', self.on_release)
        self.fig.canvas.mpl_connect('scroll_event', self.on_scroll)
        
        self.setup_plot()
        
    def create_buttons(self):
        """Создает кнопки управления"""
        # Кнопка включения/выключения режима редактирования
        ax_edit = plt.axes([0.15, 0.02, 0.2, 0.06])
        self.edit_button = Button(ax_edit, 'Режим редактирования: ВКЛ')
        self.edit_button.on_clicked(self.toggle_edit_mode)
        
        # Кнопка сохранения
        ax_save = plt.axes([0.4, 0.02, 0.15, 0.06])
        self.save_button = Button(ax_save, 'Сохранить')
        self.save_button.on_clicked(self.manual_save)
        
        # Кнопка сброса
        ax_reset = plt.axes([0.6, 0.02, 0.15, 0.06])
        self.reset_button = Button(ax_reset, 'Сброс зума')
        self.reset_button.on_clicked(self.reset_view)
        
    def toggle_edit_mode(self, event):
        """Переключает режим редактирования"""
        self.edit_mode = not self.edit_mode
        if self.edit_mode:
            self.edit_button.label.set_text('Режим редактирования: ВКЛ')
            print("Режим редактирования ВКЛЮЧЕН - клики добавляют/удаляют R-пики")
        else:
            self.edit_button.label.set_text('Режим редактирования: ВЫКЛ')
            print("Режим редактирования ВЫКЛЮЧЕН - клики не влияют на R-пики")
        self.fig.canvas.draw_idle()
        
    def manual_save(self, event):
        """Ручное сохранение"""
        save_r_peaks_to_file(self.r_peaks_indices, self.frame_numbers, self.fs)
        
    def reset_view(self, event):
        """Сбрасывает вид графика к исходному"""
        self.ax.set_xlim(self.display_time[0], self.display_time[-1])
        self.ax.set_ylim(np.min(self.display_signal), np.max(self.display_signal))
        self.fig.canvas.draw_idle()
        
    def setup_plot(self):
        """Настройка внешнего вида графика"""
        self.ax.set_xlabel('Время (секунды)')
        self.ax.set_ylabel('Амплитуда (нормализованная)')
        self.ax.set_title(f'Детекция R-пиков (fs={self.fs} Гц) - С {self.start_second} по {self.end_second} секунду\n'
                         f'Управление: ЛКМ - добавить/удалить R-пик | Колесо мыши - zoom | Перетаскивание - pan')
        self.ax.legend()
        self.ax.grid(True, alpha=0.3)
        
    def update_r_peaks_display(self):
        """Обновляет отображение R-пиков на графике"""
        # Удаляем старые точки R-пиков
        for artist in self.ax.collections:
            if hasattr(artist, '_r_peaks'):
                artist.remove()
        
        # Отображаем R-пики в текущем диапазоне
        display_peaks = [idx for idx in self.r_peaks_indices if self.start_index <= idx < self.end_index]
        
        if display_peaks:
            peak_points = self.ax.scatter(self.time[display_peaks], self.ecg_signal[display_peaks],
                                        c='red', s=100, marker='o', edgecolors='darkred', linewidths=2,
                                        label='R-пики', zorder=5)
            peak_points._r_peaks = True  # Помечаем как R-пики
    
    def update_info_text(self):
        """Обновляет информацию о количестве R-пиков"""
        total_peaks = len(self.r_peaks_indices)
        visible_peaks = len([idx for idx in self.r_peaks_indices if self.start_index <= idx < self.end_index])
        mode_status = "ВКЛ" if self.edit_mode else "ВЫКЛ"
        self.info_text.set_text(f"Всего R-пиков: {total_peaks}\nВидимых: {visible_peaks}\nРежим редактирования: {mode_status}")
    
    def find_nearest_peak(self, x, y, threshold=0.05):
        """Находит ближайший R-пик к координатам клика"""
        if not self.r_peaks_indices:
            return None
        
        # Ищем ближайший R-пик по времени
        display_peaks = [idx for idx in self.r_peaks_indices if self.start_index <= idx < self.end_index]
        
        if not display_peaks:
            return None
        
        peak_times = self.time[display_peaks]
        distances = np.abs(peak_times - x)
        
        min_idx = np.argmin(distances)
        min_distance = distances[min_idx]
        
        # Проверяем, находится ли клик достаточно близко к пику
        if min_distance < threshold:
            return display_peaks[min_idx]
        
        return None
    
    def add_r_peak(self, x, y):
        """Добавляет новый R-пик"""
        # Находим ближайшую точку данных
        ind = np.argmin(np.abs(self.display_time - x))
        global_index = self.start_index + ind
        
        # Проверяем, нет ли уже пика в этой области
        search_radius = int(0.05 * self.fs)  # 50 мс
        for peak in self.r_peaks_indices:
            if abs(peak - global_index) < search_radius:
                print(f"R-пик уже существует в этой области (кадр {self.frame_numbers[peak]})")
                return
        
        # Добавляем новый пик
        self.r_peaks_indices.append(global_index)
        self.r_peaks_indices.sort()  # Сортируем по времени
        
        # Обновляем отображение
        self.update_r_peaks_display()
        self.update_info_text()
        self.fig.canvas.draw_idle()
        
        frame = self.frame_numbers[global_index]
        time_sec = global_index / self.fs
        print(f"Добавлен R-пик: кадр {frame}, время {time_sec:.3f} сек")
        
        # Сохраняем в файл
        save_r_peaks_to_file(self.r_peaks_indices, self.frame_numbers, self.fs)
    
    def remove_r_peak(self, peak_index):
        """Удаляет R-пик"""
        if peak_index in self.r_peaks_indices:
            self.r_peaks_indices.remove(peak_index)
            
            # Обновляем отображение
            self.update_r_peaks_display()
            self.update_info_text()
            self.fig.canvas.draw_idle()
            
            frame = self.frame_numbers[peak_index]
            time_sec = peak_index / self.fs
            print(f"Удален R-пик: кадр {frame}, время {time_sec:.3f} сек")
            
            # Сохраняем в файл
            save_r_peaks_to_file(self.r_peaks_indices, self.frame_numbers, self.fs)
    
    def on_click(self, event):
        """Обработчик кликов мыши"""
        if event.inaxes != self.ax or event.button != 1:  # ЛКМ
            return
        
        x, y = event.xdata, event.ydata
        if x is None or y is None:
            return
        
        # Проверяем режим редактирования
        if not self.edit_mode:
            # Если режим редактирования выключен, начинаем перетаскивание
            self.dragging = True
            self.drag_start_x = x
            self.drag_start_y = y
            return
        
        # Режим редактирования включен - работаем с R-пиками
        # Ищем ближайший существующий R-пик
        nearest_peak = self.find_nearest_peak(x, y)
        
        if nearest_peak is not None:
            # Удаляем существующий пик
            self.remove_r_peak(nearest_peak)
        else:
            # Добавляем новый пик
            self.add_r_peak(x, y)
    
    def on_release(self, event):
        """Обработчик отпускания кнопки мыши"""
        self.dragging = False
    
    def on_scroll(self, event):
        """Обработчик скролла мыши для zoom"""
        if event.inaxes != self.ax:
            return
        
        cur_xlim = self.ax.get_xlim()
        cur_ylim = self.ax.get_ylim()
        
        xdata = event.xdata
        ydata = event.ydata
        
        if xdata is None or ydata is None:
            return
        
        scale_factor = 1.1 if event.button == 'up' else 0.9
        
        # Zoom по X
        new_width = (cur_xlim[1] - cur_xlim[0]) * scale_factor
        relx = (cur_xlim[1] - xdata) / (cur_xlim[1] - cur_xlim[0])
        new_xlim = [xdata - new_width * (1 - relx), xdata + new_width * relx]
        
        # Zoom по Y
        new_height = (cur_ylim[1] - cur_ylim[0]) * scale_factor
        rely = (cur_ylim[1] - ydata) / (cur_ylim[1] - cur_ylim[0])
        new_ylim = [ydata - new_height * (1 - rely), ydata + new_height * rely]
        
        self.ax.set_xlim(new_xlim)
        self.ax.set_ylim(new_ylim)
        self.fig.canvas.draw_idle()
    
    def on_motion(self, event):
        """Обработчик движения мыши"""
        if event.inaxes == self.ax:
            x, y = event.xdata, event.ydata
            if x is not None and y is not None:
                # Обновляем текст с координатами
                ind = np.argmin(np.abs(self.display_time - x))
                if ind < len(self.display_frames):
                    frame_val = self.display_frames[ind]
                    self.coord_text.set_text(f"Время: {x:.3f} с\nАмплитуда: {y:.3f}\nКадр: {frame_val}")
                    self.coord_text.set_visible(True)
                
                # Обновляем аннотацию
                self.update_annot(event)
                
                # Обработка перетаскивания
                if self.dragging and hasattr(self, 'drag_start_x'):
                    cur_xlim = self.ax.get_xlim()
                    cur_ylim = self.ax.get_ylim()
                    
                    dx = self.drag_start_x - x
                    dy = self.drag_start_y - y
                    
                    self.ax.set_xlim(cur_xlim[0] + dx, cur_xlim[1] + dx)
                    self.ax.set_ylim(cur_ylim[0] + dy, cur_ylim[1] + dy)
                    self.fig.canvas.draw_idle()
                    
                    self.drag_start_x = x
                    self.drag_start_y = y
        else:
            self.coord_text.set_visible(False)
            self.annot.set_visible(False)
        
        self.fig.canvas.draw_idle()
    
    def update_annot(self, event):
        """Обновляет аннотацию"""
        x, y = event.xdata, event.ydata
        if x is None or y is None:
            return
        
        # Находим ближайшую точку данных
        ind = np.argmin(np.abs(self.display_time - x))
        if ind < len(self.display_signal):
            x_val = self.display_time[ind]
            y_val = self.display_signal[ind]
            frame_val = self.display_frames[ind]
            
            # Проверяем, является ли эта точка R-пиком
            global_index = self.start_index + ind
            is_r_peak = global_index in self.r_peaks_indices
            
            self.annot.xy = (x_val, y_val)
            if is_r_peak:
                r_peak_idx = self.r_peaks_indices.index(global_index)
                self.annot.set_text(f"Кадр: {frame_val}\nВремя: {x_val:.3f} с\nАмплитуда: {y_val:.3f}\nR-пик #{r_peak_idx + 1}")
                self.annot.arrow_patch.set_color('red')
            else:
                self.annot.set_text(f"Кадр: {frame_val}\nВремя: {x_val:.3f} с\nАмплитуда: {y_val:.3f}")
                self.annot.arrow_patch.set_color('black')
            self.annot.set_visible(True)
    
    def on_leave(self, event):
        """Обработчик выхода курсора за пределы axes"""
        self.annot.set_visible(False)
        self.coord_text.set_visible(False)
        self.fig.canvas.draw_idle()
    
    def show(self):
        """Показывает график"""
        plt.show()


def plot_results(ecg_signal, r_peaks_indices, frame_numbers, fs=1000, start_second=5, end_second=10):
    """Создает интерактивный график для редактирования R-пиков"""
    editor = InteractiveRPeakEditor(ecg_signal, r_peaks_indices, frame_numbers, fs, start_second, end_second)
    editor.show()
    
    return editor.r_peaks_indices


if __name__ == "__main__":
    filename = "pointsString_80_output_sec.txt"

    try:
        r_peak_frames, r_peaks_indices, ecg_signal, frame_numbers = detect_r_peaks_from_file(filename, fs=1000)

        # Сохраняем первоначальные результаты
        save_r_peaks_to_file(r_peaks_indices, frame_numbers, fs=1000)

        # Создаем интерактивный график
        updated_r_peaks = plot_results(ecg_signal, r_peaks_indices, frame_numbers, fs=1000, start_second=0, end_second=544)

        print(f"\nФинальные результаты:")
        r_peak_frames_final = frame_numbers[updated_r_peaks]
        for i, (frame, index) in enumerate(zip(r_peak_frames_final, updated_r_peaks)):
            time_sec = index / 1000.0
            print(f"R-пик {i + 1:2d}: кадр {frame}, время {time_sec:.3f} сек")

    except FileNotFoundError:
        print(f"Файл {filename} не найден!")
    except Exception as e:
        print(f"Ошибка при обработке данных: {e}")
# Использование
process_file("pointsString_80.txt", "pointsString_80_output_sec.txt")
